# Results: Aggregate, Analyze, Export

Run this **locally** after all workers finish. Nothing here reruns experiments.

### Sections
1. **Setup** — paths, display settings
2. **Load & Check** — re-aggregate per-run CSVs, verify completeness
3. **Overview** — quick pivot tables and rankings (per alpha, never averaged)
4. **Publication Tables** — mean ± std with bold-best, LaTeX export
5. **Publication Figures** — Pareto fronts, bar charts, gradient conflict heatmaps
6. **Paper Numbers** — extract specific values for inline references
7. **Diagnostics** — knapsack training quality checks
8. **Scratchpad** — ad-hoc queries

---
## 1. Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from IPython.display import display, HTML

warnings.filterwarnings('ignore', category=FutureWarning)

# ---- Paths (local) ----
REPO_ROOT = Path.cwd()
if REPO_ROOT.name != 'DecisionFocusedMTL':
    # Try to find it
    for candidate in [Path('.'), Path('..'), Path('../..')]:
        if (candidate / 'src' / 'fair_dfl').exists():
            REPO_ROOT = candidate.resolve()
            break

HC_DIR  = REPO_ROOT / 'results' / 'final' / 'healthcare'
KN_DIR  = REPO_ROOT / 'results' / 'final' / 'knapsack_v2'
OUT_DIR = REPO_ROOT / 'results' / 'final'
TABLES_DIR = OUT_DIR / 'tables'
FIGS_DIR   = OUT_DIR / 'figures'

for d in [TABLES_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Display settings ----
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9, 'axes.labelsize': 10,
    'axes.titlesize': 11, 'legend.fontsize': 7.5,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.grid': True, 'grid.alpha': 0.3,
})

# ---- Filter settings ----
HC_HIDDEN_DIM = 64   # Only use hd=64 results (ignore hd=128 legacy runs)

print(f'Repo root:  {REPO_ROOT}')
print(f'Healthcare: {HC_DIR}')
print(f'Knapsack:   {KN_DIR}')
print(f'Output:     {OUT_DIR}')
print('Setup OK')

---
## 2. Load & Check

Always re-aggregates from per-run CSVs (ignores stale `stage_results_all.csv`).

In [ ]:
def load_all(results_dir):
    """Load all per-run CSVs from a results tree (always fresh)."""
    p = Path(results_dir)
    csvs = sorted(p.rglob('stage_results.csv'))
    if not csvs:
        print(f'WARNING: No per-run CSVs found in {p}')
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)

    # Normalize lambda column
    if 'lambda' in df.columns:
        df['lam'] = df['lambda'].fillna(0.0)
    else:
        df['lam'] = 0.0

    # Readable row labels
    def _label(r):
        m = r.get('method_label', r.get('method', '?'))
        l = r['lam']
        if m == 'FPTO'      and l == 0: return 'PTO'
        if m == 'FDFL-Scal' and l == 0: return 'DFL'
        if m in ('FPTO', 'FDFL-Scal') and l > 0:
            return f'{m} ($\\lambda$={l:g})'
        return m
    df['row_label'] = df.apply(_label, axis=1)
    return df

# Load
hc_raw = load_all(HC_DIR)
kn     = load_all(KN_DIR)

# Filter healthcare to hd=64 only
if not hc_raw.empty and 'hidden_dim' in hc_raw.columns:
    hc = hc_raw[hc_raw['hidden_dim'] == HC_HIDDEN_DIM].copy()
    n_dropped = len(hc_raw) - len(hc)
    if n_dropped > 0:
        print(f'Healthcare: dropped {n_dropped} rows with hidden_dim != {HC_HIDDEN_DIM}')
else:
    hc = hc_raw.copy()

# Save fresh aggregates
for df, d in [(hc, HC_DIR), (kn, KN_DIR)]:
    if not df.empty:
        df.to_csv(Path(d) / 'stage_results_all.csv', index=False)

print(f'\nHealthcare: {len(hc)} rows (hd={HC_HIDDEN_DIM})')
print(f'Knapsack:   {len(kn)} rows')

In [ ]:
EXPECTED_METHODS = ['FPTO', 'SAA', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
EXPECTED_SEEDS = [11, 22, 33, 44, 55]

def check(df, name, extra_cols):
    if df.empty:
        print(f'{name}: NO DATA'); return
    print(f'\n=== {name}: {len(df)} rows ===')
    if 'method_label' in df.columns:
        found = set(df['method_label'].unique())
        missing = set(EXPECTED_METHODS) - found
        status = ' (all present)' if not missing else f'  MISSING: {missing}'
        print(f'  Methods: {sorted(found)}{status}')
    if 'seed' in df.columns:
        seeds = sorted(df['seed'].unique())
        print(f'  Seeds: {seeds}' + (' (OK)' if seeds == EXPECTED_SEEDS else f' EXPECTED {EXPECTED_SEEDS}'))
    for col in extra_cols:
        if col in df.columns:
            print(f'  {col}: {sorted(df[col].unique())}')
    # method x lambda count
    if 'method_label' in df.columns:
        print(f'\n  Runs per (method_label, lambda):')
        print(df.groupby(['method_label', 'lam']).size().to_string())

check(hc, 'Healthcare', ['alpha_fair', 'hidden_dim', 'lam'])
check(kn, 'Knapsack',   ['alpha_fair', 'unfairness_level', 'lam'])

---
## 3. Overview

In [ ]:
# ---- Reusable constants ----
ROW_ORDER = [
    'PTO', 'FPTO ($\\lambda$=0.5)', 'FPTO ($\\lambda$=1)', 'FPTO ($\\lambda$=5)',
    'SAA', 'WDRO',
    'DFL', 'FDFL-Scal ($\\lambda$=0.5)', 'FDFL-Scal ($\\lambda$=1)', 'FDFL-Scal ($\\lambda$=5)',
    'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad',
]

METRIC_COLS = ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
METRIC_NAMES = {
    'test_regret_normalized': 'Norm. Regret',
    'test_fairness': 'Pred. Fairness Viol.',
    'test_pred_mse': 'Pred. MSE',
    'avg_cos_dec_pred': 'cos(dec,pred)',
    'avg_cos_dec_fair': 'cos(dec,fair)',
    'avg_cos_pred_fair': 'cos(pred,fair)',
}

COLORS = {
    'PTO': '#636363', 'SAA': '#e6550d', 'WDRO': '#756bb1',
    'DFL': '#c49c94',
    'FDFL-PCGrad': '#2ca02c', 'FDFL-MGDA': '#d62728', 'FDFL-CAGrad': '#9467bd',
}
for l in ['0.5', '1', '5']:
    COLORS[f'FPTO ($\\lambda$={l})'] = '#1f77b4'
    COLORS[f'FDFL-Scal ($\\lambda$={l})'] = '#ff7f0e'

MARKERS = {
    'PTO': 'o', 'SAA': 'D', 'WDRO': 'p', 'DFL': 's',
    'FDFL-PCGrad': '*', 'FDFL-MGDA': 'h', 'FDFL-CAGrad': 'd',
}
for l in ['0.5', '1', '5']:
    MARKERS[f'FPTO ($\\lambda$={l})'] = '^'
    MARKERS[f'FDFL-Scal ($\\lambda$={l})'] = 'v'

print(f'ROW_ORDER has {len(ROW_ORDER)} entries')

In [ ]:
# ---- Helper functions ----

def _reindex(df_or_series, row_order=ROW_ORDER):
    """Reindex to canonical row order, dropping missing."""
    order = [r for r in row_order if r in df_or_series.index]
    return df_or_series.reindex(order)


def pivot(df, metric, col_col='alpha_fair'):
    """Quick pivot: methods x conditions (mean over seeds)."""
    piv = df.pivot_table(index='row_label', columns=col_col,
                         values=metric, aggfunc='mean')
    return _reindex(piv)


def mean_std_table(df, panel_col='alpha_fair', metrics=None):
    """Publication-style mean +/- std table, separate columns per panel."""
    metrics = metrics or METRIC_COLS
    valid = [m for m in metrics if m in df.columns]
    panels = sorted(df[panel_col].unique())
    grouped = df.groupby(['row_label', panel_col])
    means = grouped[valid].mean()
    stds  = grouped[valid].std()

    rows = []
    for method in ROW_ORDER:
        if method not in df['row_label'].values:
            continue
        row = {'Method': method}
        for panel in panels:
            for m in valid:
                col = f'{METRIC_NAMES.get(m,m)}|{panel}'
                try:
                    mu = means.loc[(method, panel), m]
                    sd = stds.loc[(method, panel), m]
                    row[col] = f'{mu:.4f} +/- {sd:.4f}'
                except KeyError:
                    row[col] = '--'
        rows.append(row)
    return pd.DataFrame(rows).set_index('Method')


def bold_best_table(df, panel_col='alpha_fair', metrics=None, lower_better=True):
    """HTML table with best values bolded per metric per panel."""
    metrics = metrics or METRIC_COLS
    valid = [m for m in metrics if m in df.columns]
    panels = sorted(df[panel_col].unique())
    means = df.groupby(['row_label', panel_col])[valid].mean()
    stds  = df.groupby(['row_label', panel_col])[valid].std()

    # Find best per (panel, metric)
    best = {}
    for panel in panels:
        for m in valid:
            vals = {}
            for method in ROW_ORDER:
                try: vals[method] = means.loc[(method, panel), m]
                except KeyError: pass
            if vals:
                best[(panel, m)] = min(vals, key=vals.get) if lower_better else max(vals, key=vals.get)

    rows_html = []
    for method in ROW_ORDER:
        if method not in df['row_label'].values:
            continue
        cells = [f'<td style="text-align:left"><b>{method}</b></td>']
        for panel in panels:
            for m in valid:
                try:
                    mu = means.loc[(method, panel), m]
                    sd = stds.loc[(method, panel), m]
                    txt = f'{mu:.4f} \u00b1 {sd:.4f}'
                    if best.get((panel, m)) == method:
                        txt = f'<b>{txt}</b>'
                    cells.append(f'<td>{txt}</td>')
                except KeyError:
                    cells.append('<td>--</td>')
        rows_html.append('<tr>' + ''.join(cells) + '</tr>')

    # Header
    hdr1 = '<th></th>'
    hdr2 = '<th>Method</th>'
    for panel in panels:
        hdr1 += f'<th colspan="{len(valid)}" style="text-align:center">{panel_col}={panel}</th>'
        for m in valid:
            hdr2 += f'<th>{METRIC_NAMES.get(m,m)}</th>'

    html = f'<table><thead><tr>{hdr1}</tr><tr>{hdr2}</tr></thead><tbody>'
    html += ''.join(rows_html)
    html += '</tbody></table>'
    return HTML(html)


def ranks(df, condition_cols, metrics=None):
    """Average rank per method across conditions (1=best)."""
    metrics = metrics or METRIC_COLS
    valid = [m for m in metrics if m in df.columns]
    all_r = []
    for _, grp in df.groupby(condition_cols):
        mm = grp.groupby('row_label')[valid].mean()
        for m in valid:
            ranked = mm[m].rank(ascending=True)
            for method, rank in ranked.items():
                all_r.append({'method': method, 'metric': m, 'rank': rank})
    if not all_r:
        return pd.DataFrame()
    avg = pd.DataFrame(all_r).groupby(['method', 'metric'])['rank'].mean().unstack()
    avg.columns = [METRIC_NAMES.get(m, m) for m in avg.columns]
    avg['Overall'] = avg.mean(axis=1)
    return _reindex(avg).sort_values('Overall')


def get_val(df, method, metric, alpha=None, uf=None):
    """Extract mean/std/n for a specific slice. For inline paper refs."""
    mask = df['row_label'] == method
    if alpha is not None:
        mask &= df['alpha_fair'] == alpha
    if uf is not None:
        mask &= df['unfairness_level'] == uf
    v = df.loc[mask, metric].dropna()
    return {'mean': v.mean(), 'std': v.std(), 'n': len(v)} if len(v) else None


def to_latex(mean, std, fmt='.4f', bold=False):
    s = f'${mean:{fmt}} \\pm {std:{fmt}}$'
    return f'\\textbf{{{s}}}' if bold else s


print('Helpers loaded')

In [ ]:
# ---- Healthcare overview ----
if not hc.empty:
    print('Healthcare: Normalized Decision Regret (mean over seeds)')
    display(pivot(hc, 'test_regret_normalized'))
    print('\nHealthcare: Prediction Fairness Violation')
    display(pivot(hc, 'test_fairness'))
    print('\nHealthcare: Prediction MSE')
    display(pivot(hc, 'test_pred_mse'))

In [ ]:
# ---- Knapsack overview (per alpha) ----
if not kn.empty and 'unfairness_level' in kn.columns:
    for alpha in sorted(kn['alpha_fair'].unique()):
        sub = kn[kn['alpha_fair'] == alpha]
        print(f'\n=== Knapsack alpha={alpha} ===')
        print('Norm. Regret:')
        display(pivot(sub, 'test_regret_normalized', col_col='unfairness_level'))
        print('Fairness Violation:')
        display(pivot(sub, 'test_fairness', col_col='unfairness_level'))

In [ ]:
# ---- Rankings ----
if not hc.empty:
    print('Healthcare Rankings (1=best):')
    display(ranks(hc, condition_cols=['alpha_fair']))

if not kn.empty:
    cond = ['alpha_fair']
    if 'unfairness_level' in kn.columns:
        cond.append('unfairness_level')
    print('\nKnapsack Rankings (1=best):')
    display(ranks(kn, condition_cols=cond))

---
## 4. Publication Tables

In [ ]:
# ---- Table 1: Healthcare (bold best) ----
if not hc.empty:
    print('Table 1: Healthcare')
    display(bold_best_table(hc, panel_col='alpha_fair'))
    print('\nPlain text version:')
    display(mean_std_table(hc, panel_col='alpha_fair'))

In [ ]:
# ---- Table 2: Knapsack per alpha ----
if not kn.empty and 'unfairness_level' in kn.columns:
    for alpha in sorted(kn['alpha_fair'].unique()):
        sub = kn[kn['alpha_fair'] == alpha]
        print(f'\nTable 2 (alpha={alpha}): Knapsack')
        display(bold_best_table(sub, panel_col='unfairness_level'))

In [ ]:
# ---- LaTeX export ----

def _latex_table(df, panel_col, metrics=None, caption='', label=''):
    """Generate LaTeX tabular string with bold-best."""
    metrics = metrics or METRIC_COLS
    valid = [m for m in metrics if m in df.columns]
    panels = sorted(df[panel_col].unique())
    means = df.groupby(['row_label', panel_col])[valid].mean()
    stds  = df.groupby(['row_label', panel_col])[valid].std()

    # Best per (panel, metric)
    best = {}
    for panel in panels:
        for m in valid:
            vals = {}
            for method in ROW_ORDER:
                try: vals[method] = means.loc[(method, panel), m]
                except: pass
            if vals:
                best[(panel, m)] = min(vals, key=vals.get)

    n_met = len(valid)
    n_panels = len(panels)
    col_spec = 'l' + '|' + '|'.join(['c' * n_met] * n_panels)

    lines = []
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    if caption: lines.append(f'\\caption{{{caption}}}')
    if label:   lines.append(f'\\label{{{label}}}')
    lines.append(f'\\begin{{tabular}}{{{col_spec}}}')
    lines.append(r'\toprule')

    # Header row 1: panel names
    hdr1 = ' & '.join([''] + [f'\\multicolumn{{{n_met}}}{{c}}{{{panel_col}={p}}}' for p in panels])
    lines.append(hdr1 + r' \\')
    # Header row 2: metric names
    hdr2_parts = ['Method']
    for p in panels:
        for m in valid:
            hdr2_parts.append(METRIC_NAMES.get(m, m))
    lines.append(' & '.join(hdr2_parts) + r' \\')
    lines.append(r'\midrule')

    for method in ROW_ORDER:
        if method not in df['row_label'].values: continue
        # Escape $ for latex method names
        method_tex = method.replace('$\\lambda$', r'$\lambda$')
        cells = [method_tex]
        for panel in panels:
            for m in valid:
                try:
                    mu = means.loc[(method, panel), m]
                    sd = stds.loc[(method, panel), m]
                    txt = f'{mu:.4f}$\\pm${sd:.4f}'
                    if best.get((panel, m)) == method:
                        txt = f'\\textbf{{{txt}}}'
                    cells.append(txt)
                except:
                    cells.append('--')
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


# Generate and save
if not hc.empty:
    tex = _latex_table(hc, 'alpha_fair',
                       caption='Healthcare Resource Allocation Results',
                       label='tab:healthcare')
    (TABLES_DIR / 'table_healthcare.tex').write_text(tex)
    print('Saved table_healthcare.tex')

if not kn.empty and 'unfairness_level' in kn.columns:
    for alpha in sorted(kn['alpha_fair'].unique()):
        sub = kn[kn['alpha_fair'] == alpha]
        tex = _latex_table(sub, 'unfairness_level',
                           caption=f'Knapsack Results ($\\alpha={alpha}$)',
                           label=f'tab:knapsack_a{alpha}')
        fname = f'table_knapsack_alpha{alpha}.tex'
        (TABLES_DIR / fname).write_text(tex)
        print(f'Saved {fname}')

---
## 5. Publication Figures

In [ ]:
# ---- Fig 1: Pareto front (Healthcare) ----
if not hc.empty:
    alphas = sorted(hc['alpha_fair'].unique())
    fig, axes = plt.subplots(1, len(alphas), figsize=(5.5 * len(alphas), 4.5))
    if len(alphas) == 1: axes = [axes]

    for ax, alpha in zip(axes, alphas):
        sub = hc[hc['alpha_fair'] == alpha]
        means = sub.groupby('row_label')[['test_regret_normalized', 'test_fairness']].mean()
        for method in ROW_ORDER:
            if method not in means.index: continue
            x = means.loc[method, 'test_regret_normalized']
            y = means.loc[method, 'test_fairness']
            if np.isnan(x) or np.isnan(y): continue
            ax.scatter(x, y,
                       c=COLORS.get(method, '#333'),
                       marker=MARKERS.get(method, 'o'),
                       s=90, edgecolors='k', linewidths=0.5,
                       label=method, zorder=5)
        ax.set_xlabel('Normalized Decision Regret')
        ax.set_ylabel('Prediction Fairness Violation')
        ax.set_title(f'Healthcare ($\\alpha$ = {alpha})')
        ax.legend(fontsize=5.5, loc='best', ncol=2)

    plt.tight_layout()
    fig.savefig(FIGS_DIR / 'fig1_pareto_healthcare.pdf')
    plt.show()
    print(f'Saved {FIGS_DIR / "fig1_pareto_healthcare.pdf"}')

In [ ]:
# ---- Fig 2: Regret by unfairness level (Knapsack) ----
if not kn.empty and 'unfairness_level' in kn.columns:
    focus = ['PTO', 'FPTO ($\\lambda$=1)', 'SAA', 'DFL',
             'FDFL-Scal ($\\lambda$=1)', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
    alphas = sorted(kn['alpha_fair'].unique())
    fig, axes = plt.subplots(1, len(alphas), figsize=(6.5 * len(alphas), 4.5))
    if len(alphas) == 1: axes = [axes]

    for ax, alpha in zip(axes, alphas):
        sub = kn[(kn['alpha_fair'] == alpha) & kn['row_label'].isin(focus)]
        piv = sub.pivot_table(index='row_label', columns='unfairness_level',
                              values='test_regret_normalized', aggfunc='mean')
        piv = piv.reindex([m for m in focus if m in piv.index])
        cols = [c for c in ['mild', 'medium', 'high'] if c in piv.columns]
        piv[cols].plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(f'Knapsack ($\\alpha$ = {alpha})')
        ax.set_ylabel('Norm. Regret')
        ax.tick_params(axis='x', rotation=40)
        ax.legend(title='Unfairness')

    plt.tight_layout()
    fig.savefig(FIGS_DIR / 'fig2_regret_by_unfairness.pdf')
    plt.show()

In [ ]:
# ---- Fig 3: Gradient conflict heatmap (Knapsack MOO methods) ----
if not kn.empty and 'unfairness_level' in kn.columns:
    cos_cols = [c for c in ['avg_cos_dec_pred', 'avg_cos_dec_fair', 'avg_cos_pred_fair']
               if c in kn.columns]
    moo_methods = ['FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
    moo = kn[kn['row_label'].isin(moo_methods)]

    if not moo.empty and cos_cols:
        fig, axes = plt.subplots(1, len(cos_cols), figsize=(4.2 * len(cos_cols), 3.5))
        if len(cos_cols) == 1: axes = [axes]

        for ax, col in zip(axes, cos_cols):
            piv = moo.pivot_table(index='row_label', columns='unfairness_level',
                                  values=col, aggfunc='mean')
            order = [c for c in ['mild', 'medium', 'high'] if c in piv.columns]
            piv = piv[order]
            im = ax.imshow(piv.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
            for i in range(piv.shape[0]):
                for j in range(piv.shape[1]):
                    v = piv.values[i, j]
                    if not np.isnan(v):
                        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                                fontsize=8,
                                color='white' if abs(v) > 0.4 else 'black')
            ax.set_xticks(range(len(order)))
            ax.set_xticklabels(order)
            ax.set_yticks(range(piv.shape[0]))
            ax.set_yticklabels(piv.index)
            ax.set_title(METRIC_NAMES.get(col, col))

        fig.colorbar(im, ax=axes, shrink=0.8, label='Cosine similarity')
        plt.tight_layout()
        fig.savefig(FIGS_DIR / 'fig3_gradient_conflict.pdf')
        plt.show()

---
## 6. Paper Numbers

In [ ]:
# ---- Key numbers ----
focus_methods = ['PTO', 'FPTO ($\\lambda$=1)', 'SAA', 'WDRO',
                 'DFL', 'FDFL-Scal ($\\lambda$=1)',
                 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']

if not hc.empty:
    print('=== Healthcare Key Metrics ===')
    for method in focus_methods:
        for alpha in [0.5, 2.0]:
            r = get_val(hc, method, 'test_regret_normalized', alpha=alpha)
            if r:
                print(f'  {method:35s} a={alpha}: {r["mean"]:.4f} +/- {r["std"]:.4f} (n={r["n"]})')

In [ ]:
# ---- % Improvement over PTO baseline ----
if not hc.empty:
    print('=== % Improvement over PTO (Healthcare) ===')
    for alpha in [0.5, 2.0]:
        base = get_val(hc, 'PTO', 'test_regret_normalized', alpha=alpha)
        if not base: continue
        print(f'\nalpha={alpha} (PTO regret = {base["mean"]:.4f}):')
        for m in focus_methods:
            if m == 'PTO': continue
            r = get_val(hc, m, 'test_regret_normalized', alpha=alpha)
            if r:
                pct = (base['mean'] - r['mean']) / base['mean'] * 100
                print(f'  {m:35s}: {pct:+.1f}% ({r["mean"]:.4f})')

In [ ]:
# ---- LaTeX copy-paste snippets ----
if not hc.empty:
    print('=== LaTeX snippets (Healthcare) ===')
    for m in focus_methods:
        for a in [0.5, 2.0]:
            r = get_val(hc, m, 'test_regret_normalized', alpha=a)
            if r:
                print(f'% {m}, alpha={a}:  {to_latex(r["mean"], r["std"])}')

---
## 7. Diagnostics

Check training quality: Are methods actually learning? Is the SPSA gradient effective?

In [ ]:
# ---- Metric spread: how much do methods differ? ----
for name, df in [('Healthcare', hc), ('Knapsack', kn)]:
    if df.empty: continue
    print(f'\n=== {name}: Metric range across methods ===')
    panel_col = 'alpha_fair'
    for alpha in sorted(df['alpha_fair'].unique()):
        sub = df[df['alpha_fair'] == alpha]
        if 'unfairness_level' in sub.columns:
            for uf in sorted(sub['unfairness_level'].unique()):
                ss = sub[sub['unfairness_level'] == uf]
                g = ss.groupby('row_label')[METRIC_COLS].mean()
                for m in METRIC_COLS:
                    if m in g.columns:
                        lo, hi = g[m].min(), g[m].max()
                        spread = (hi - lo) / lo * 100 if lo > 0 else 0
                        print(f'  a={alpha} uf={uf:6s} {METRIC_NAMES.get(m,m):25s}: '
                              f'range=[{lo:.4f}, {hi:.4f}] spread={spread:.1f}%')
        else:
            g = sub.groupby('row_label')[METRIC_COLS].mean()
            for m in METRIC_COLS:
                if m in g.columns:
                    lo, hi = g[m].min(), g[m].max()
                    spread = (hi - lo) / lo * 100 if lo > 0 else 0
                    print(f'  a={alpha} {METRIC_NAMES.get(m,m):25s}: '
                          f'range=[{lo:.4f}, {hi:.4f}] spread={spread:.1f}%')

In [ ]:
# ---- SAA comparison: Is the MLP beating the mean predictor? ----
for name, df in [('Healthcare', hc), ('Knapsack', kn)]:
    if df.empty: continue
    print(f'\n=== {name}: SAA vs Trained Methods (MSE) ===')
    saa_mse = df[df['row_label'] == 'SAA'].groupby('alpha_fair')['test_pred_mse'].mean()
    for alpha in sorted(df['alpha_fair'].unique()):
        if alpha not in saa_mse.index: continue
        saa_val = saa_mse[alpha]
        sub = df[(df['alpha_fair'] == alpha) & (df['row_label'] != 'SAA')]
        trained_mse = sub.groupby('row_label')['test_pred_mse'].mean()
        print(f'  alpha={alpha}: SAA MSE = {saa_val:.4f}')
        for m in ROW_ORDER:
            if m in trained_mse.index and m != 'SAA':
                v = trained_mse[m]
                pct = (v - saa_val) / saa_val * 100
                flag = ' *** WORSE THAN SAA ***' if v > saa_val else ''
                print(f'    {m:35s}: {v:.4f} ({pct:+.1f}%){flag}')

In [ ]:
# ---- Gradient norms: Are gradients meaningful or near-zero? ----
for name, df in [('Healthcare', hc), ('Knapsack', kn)]:
    if df.empty or 'avg_grad_norm_combined' not in df.columns: continue
    print(f'\n=== {name}: Gradient norms (avg over seeds) ===')
    g = df.groupby('row_label')[['avg_grad_norm_combined', 'grad_norm_min',
                                  'grad_norm_median', 'grad_norm_max',
                                  'solver_calls_train']].mean()
    display(_reindex(g))

---
## 8. Scratchpad

Use `hc` and `kn` DataFrames + helper functions. Modify freely.

In [ ]:
# Q: Which method wins at alpha=2.0, high unfairness?
if not kn.empty:
    print(kn[(kn['alpha_fair']==2.0) & (kn['unfairness_level']=='high')]
          .groupby('row_label')['test_regret_normalized'].mean().sort_values())

In [ ]:
# Q: MOO vs best scalarized across all conditions (Healthcare)?
if not hc.empty:
    moo_avg = hc[hc['row_label'].isin(['FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad'])].groupby('alpha_fair')['test_regret_normalized'].mean()
    scal_best = hc[hc['method_label']=='FDFL-Scal'].groupby('alpha_fair')['test_regret_normalized'].min()
    display(pd.DataFrame({'MOO avg': moo_avg, 'Best scal': scal_best, 'Diff %': (scal_best-moo_avg)/scal_best*100}))

In [ ]:
# Q: Does fairness violation increase with unfairness level?
if not kn.empty:
    display(kn.groupby(['row_label','unfairness_level'])['test_fairness'].mean()
            .unstack().reindex([r for r in ROW_ORDER if r in kn['row_label'].values]))

In [ ]:
# Your query here


In [ ]:
# ---- List all output files ----
print('\nAll output files:')
for d in [TABLES_DIR, FIGS_DIR]:
    if d.exists():
        for f in sorted(d.rglob('*')):
            if f.is_file():
                print(f'  {f.relative_to(OUT_DIR)} ({f.stat().st_size/1024:.0f} KB)')